In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:
def _add_gas_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Gas price proxy features derived from Henry Hub daily spot price.

    Gas is the marginal fuel in NSW during shoulder and peak periods.
    Movements in the gas price shift the short-run marginal cost of
    open-cycle and combined-cycle gas turbines, directly lifting or
    suppressing the price stack.  Key signals:

      gas_price_hh      — current daily price (USD/mmbtu, broadcast to 30-min)
      gas_srmc_proxy    — approximate SRMC of OCGT (gas_price * 10 USD/MWh)
      gas_lags          — 1d / 7d / 30d lags for short/medium-term trend     gas_roc_7d/30d    — rate-of-change (week-on-week, month-on-month)
      gas_rmean_7d/30d  — rolling averages (trend regime)
      gas_regime_high   — binary flag for top-quartile gas price periods
      gas_spread_vs_30d — deviation from 30-day mean (spike relative to trend)

    No-op if gas_price_hh is absent or all-NaN.
    """
    df = df.copy()
    OCGT_HEAT_RATE = 10.0       # approximate GJ/MWh (or mmbtu/MWh) for OCGT
    HIGH_GAS_QUANTILE = 0.75    # "high gas price" regime threshold

    if "gas_price_hh" not in df.columns or df["gas_price_hh"].isna().all():
        return df

    g = df["gas_price_hh"]
    df["gas_price_hh"] = g.astype(np.float32)

    # Short-run marginal cost proxy: gas_price × heat-rate gives USD/MWh
    # (units differ from AUD/MWh, but the model learns the scaling)
    df["gas_srmc_proxy"] = (g * OCGT_HEAT_RATE).astype(np.float32)

    # Lags (daily → 48 intervals, weekly → 336, monthly → 1440)
    for lag, name in [(48, "1d"), (96, "2d"), (336, "7d"), (1440, "30d")]:
        df[f"gas_lag_{name}"] = g.shift(lag).astype(np.float32)

    # Rolling averages — regime filter
    g_rmean_7  = g.rolling(336,  min_periods=1).mean()
    g_rmean_30 = g.rolling(1440, min_periods=1).mean()
    df["gas_rmean_7d"]  = g_rmean_7.astype(np.float32)
    df["gas_rmean_30d"] = g_rmean_30.astype(np.float32)

    # Rate of change: current vs 7-day-ago and 30-day-ago rolling mean
    df["gas_roc_7d"]  = ((g - g.shift(336))  / (g.shift(336)  + 0.01)).clip(-2, 2).astype(np.float32)
    df["gas_roc_30d"] = ((g - g.shift(1440)) / (g.shift(1440) + 0.01)).clip(-2, 2).astype(np.float32)

    # Deviation from 30-day trend (spike relative to recent baseline)
    df["gas_spread_vs_30d"] = (g - g_rmean_30).astype(np.float32)

    # Gas price percentile rank within recent windows (is gas expensive vs norms?)
    df["gas_pctrank_336"]  = g.rolling(336,  min_periods=24).rank(pct=True).astype(np.float32)
    df["gas_pctrank_1440"] = g.rolling(1440, min_periods=336).rank(pct=True).astype(np.float32)

    # High-gas-price regime flag (set when price is in top quartile of last 6 months)
    g_quantile = g.rolling(8640, min_periods=336).quantile(HIGH_GAS_QUANTILE)
    df["gas_regime_high"] = (g > g_quantile).astype(np.float32)

    return df